# T5 · The loop, as the run ran it

**Tiny Transformer, Unit 2, module T5.** T1 drew the windows, T2 scored them, T3 ran the way back and T4 turned each gradient into
a step. `train.py` joins them into one turn, and turns 100,000 times. Here you run one turn in the run's order, leave the old
gradients in, make a new AdamW every turn, count what carries on, open the file the run saved, and run a second seed.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T5-training-step.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook T5-training-step.ipynb`.

**Computed here:** the run's first turns, replayed from its seed with one change at a time, and a second seed's first 5,000.
**Checked against the run:** its recorded first batch, its log, the file it saved, and the course's numbers, which the notebook
asserts.

Laid out like the page: two sections, four chapters, and under each chapter its page's three steps (Picture it, Watch it in our
run, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import hashlib, math, os, urllib.request
import torch

torch.set_num_threads(4)     # as the run did: the same count adds each sum in the same order (see T5.4)
print(torch.__version__)

2.10.0+cpu


**The text, the model and this module's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`, copied
in whole; then Unit 2's teaching functions, `code/training_steps.py`, copied in whole: T5's are the last four.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
#@title Unit 2's teaching functions: code/training_steps.py, copied whole (run me) { display-mode: "form" }
"""Unit 2 · Inside the learning loop: the recorded run's training, station by station, done explicitly.

The teaching functions of Unit 2 (each T notebook carries this file whole, and the chapter pages light its lines; the first
line is a Colab form title, so the notebooks fold this cell and no chapter's question is answered by its code early). They sit
beside M6's code/training_loop.py, with its names (get_batch, loss_of, train_step, every_window, average_loss), and repeat the
recorded run's own code, training/prompter/train.py: test_training_steps.py checks them against it and against the run's
numbers (training/prompter/unit2.py's export, prompter-unit2.json).
"""
import math
import os
import re

import torch


# ---------------------------------------------------------------------------------------------------- T1 · picking the batches

def check_starts(n_train, n_val, batches=100, B=64, T=32, seed=2024):
    """The checks' fixed windows, as train.py draws them: 100 batches of starts in the training text, then 100 in the held-back
    text."""
    g = torch.Generator().manual_seed(seed)                       # the checks' generator
    train = torch.stack([torch.randint(n_train - T - 1, (B,), generator=g) for _ in range(batches)])   # [batches×B]
    val = torch.stack([torch.randint(n_val - T - 1, (B,), generator=g) for _ in range(batches)])       # [batches×B]
    return train, val


def run_starts(n, steps=100_000, B=64, T=32, seed=1337, make_model=None):
    """Every step's B window starts, as the recorded run drew them (train.py): the main random stream, started by the seed,
    first fills the model's random starting numbers (make_model builds the model), then draws B starts a step."""
    torch.manual_seed(seed)                                       # the main stream: starting numbers, then every batch
    if make_model is not None:
        make_model()                                              # first: the model's random starting numbers
    return torch.stack([torch.randint(n - T - 1, (B,)) for _ in range(steps)])   # [steps×B]  then B starts every step


def times_drawn(starts, n, T=32):
    """How many times each possible start was drawn. randint(n - T - 1) never draws its limit: starts run 0 … n - T - 2."""
    return torch.bincount(starts.flatten(), minlength=n - T - 1)  # [n−T−1]   one count per possible start


def times_asked(starts, n, T=32):
    """How many times each character of the text was a right answer: a window from start i asks about i + 1 … i + T."""
    drawn = times_drawn(starts, n, T)                             # [n−T−1]
    change = torch.zeros(n + 1, dtype=torch.long)
    change[1:n - T] += drawn                                      # a window from i starts asking at i + 1 …
    change[T + 1:n] -= drawn                                      # … and stops after i + T
    return change.cumsum(0)[:n]                                   # [n]       one count per character


def times_through(n, steps=100_000, B=64, T=32):
    """How many times over the run reads its text: every step asks B × T questions, one per right answer."""
    return steps * B * T / n                                      # answers ÷ characters


# ---------------------------------------------------------------------------------------------------- T2 · the loss

def all_at_once(model, x, y):
    """A batch's right answers all at once: their probabilities multiplied, in the model's 32-bit numbers, and their costs,
    minus each one's log, added and averaged (the loss)."""
    probs = torch.softmax(model(x)[0], dim=-1)                    # [B×T×65]  probabilities, as M6's loss_of makes them
    p = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)             # [B×T]     each right answer's probability
    product = p.prod()                                            # []        all of them together: too small for 32 bits, 0.0
    total = -p.log().sum()                                        # []        their costs, added
    return product, total, total / p.numel()                      # the product, the sum, and the average: the loss


def loss_from_scores(model, x, y):
    """The loss as model.py's F.cross_entropy works it out, straight from the scores: for each answer, the log of the total of
    e to every score, minus the right answer's own score. No probability is ever made."""
    scores = model(x)[0]                                          # [B×T×65]  a score for every character
    right = scores.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [B×T]     each right answer's score
    return (scores.logsumexp(-1) - right).mean()                  # []        minus the log of each probability, averaged


def answer_costs(model, ids, T=32, per=256):
    """Every right answer's probability over a whole text, and whether it was the top pick, read as the final score reads it
    (train.py full_loss): windows side by side, every character scored once but the first and the last few, which fill no
    window."""
    n = len(range(0, len(ids) - T - 1, T))                        # how many whole windows fit
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×T] each: the inputs, and the targets moved one along
    p, top = [], []
    with torch.no_grad():
        for xb, yb in zip(x.split(per), y.split(per)):
            probs = torch.softmax(model(xb)[0], dim=-1)           # [256×T×65]
            p.append(probs.gather(-1, yb.unsqueeze(-1)).squeeze(-1))   # [256×T]  the right answer's probability
            top.append(probs.argmax(-1) == yb)                    # [256×T]  was it the top pick?
    return torch.cat(p).flatten(), torch.cat(top).flatten()       # [n·T] each


def share_of_loss(p, below=0.01):
    """The answers given a probability under `below`: their share of all the answers, and their share of the loss."""
    cost = -p.log()                                               # each answer's cost
    rare = p < below
    return rare.float().mean().item(), (cost[rare].sum() / cost.sum()).item()


def sure_or_right(p, top):
    """Accuracy, the share of answers whose top pick is right; and the share right but unsure: top pick, yet under one half."""
    return top.float().mean().item(), (top & (p < 0.5)).float().mean().item()


def choices(loss):
    """A loss read backwards: an even guess among n characters costs ln n on every answer, so n = e to the loss."""
    return math.exp(loss)


# ---------------------------------------------------------------------------------------------------- T3 · the way back

def bias_gradient(probs, y):
    """Every readout bias's gradient, by its rule. A bias adds to one character's score at every place, so its gradient is that
    character's probability averaged over the places, minus its share of the right answers."""
    V = probs.size(-1)
    predicted = probs.reshape(-1, V).mean(0)                      # [65]  each character's average probability
    right = torch.bincount(y.flatten(), minlength=V).float() / y.numel()   # [65]  its share of the right answers
    return predicted - right                                      # [65]  average probability, minus share of right answers


def nudge(model, x, y, param, i, h=1e-3):
    """One number's gradient without backpropagation: nudge it up by h and run the batch, then down by h and run it again, and
    take the slope through the two losses. The number is put back as it was."""
    flat = param.data.view(-1)
    old = flat[i].item()
    with torch.no_grad():
        flat[i] = old + h
        up = model(x, y)[1].item()                                # the loss, nudged up
        flat[i] = old - h
        down = model(x, y)[1].item()                              # the loss, nudged down
        flat[i] = old                                             # put back
    return (up - down) / (2 * h)


def grad_twice(model, x, y, param):
    """backward fills every learned number's .grad in one pass, and moves none of them. Run it again without clearing, and the
    gradients add up: why train.py clears them first, with opt.zero_grad()."""
    model.zero_grad()
    model(x, y)[1].backward()                                     # every .grad filled; no number moved
    once = param.grad.clone()
    model(x, y)[1].backward()                                     # again, without clearing
    return once, param.grad.clone()                               # the second: twice the first


def backward_sizes(model, x, y):
    """The gradient on its way back through model.py's pass, part by part: its size (all its numbers measured together) at the
    scores, at the block's top, into each add's branch, below each add, and at the rows the tables made."""
    blk = model.blocks[0]
    rows = model.tok_emb(x) + model.pos_emb(torch.arange(x.size(1)))   # [B×T×28]  token row + position row
    att = blk.attn(blk.ln1(rows))                                 # the attention's branch
    mid = rows + att                                              # its add
    ff = blk.ff(blk.ln2(mid))                                     # the feed-forward's branch
    top = mid + ff                                                # its add: the block's top
    scores = model.ln_f(top) @ model.tok_emb.weight.T + model.head_bias   # [B×T×65]  the readout
    loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
    parts = {"scores": scores, "top": top, "ff_branch": ff, "meet": mid, "attn_branch": att, "rows": rows}
    grads = torch.autograd.grad(loss, list(parts.values()))       # every part's gradient, in one pass back
    return {name: g.norm().item() for name, g in zip(parts, grads)}, dict(zip(parts, grads))


def table_gradient_by_use(model, x, y):
    """The token table's gradient, one job at a time: read in at the start (each input's entry), and reused at the readout (scoring
    every character). Holding one use still (detach) leaves the other's push; the two add up to the whole gradient."""
    W = model.tok_emb.weight                                      # [65×28]

    def push(use):
        rows = (W if use == "in" else W.detach())[x] + model.pos_emb(torch.arange(x.size(1)))
        for b in model.blocks:
            rows = b(rows)
        scores = model.ln_f(rows) @ (W if use == "out" else W.detach()).T + model.head_bias
        loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
        return torch.autograd.grad(loss, W)[0]                    # [65×28]

    return push("in"), push("out")


# ---------------------------------------------------------------------------------------------------- T4 · AdamW

def adamw_step(p, g, m, v, t, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step for any tensor of numbers, as torch.optim.AdamW takes it on PyTorch's defaults (train.py passes only the
    learning rate). m and v are the running averages of the pushes and of their squares, both 0 before step 1; t counts the
    steps from 1. Returns the moved numbers and the two averages."""
    b1, b2 = betas
    p = p * (1 - lr * weight_decay)                               # the W: weight decay, a pull toward 0 (T4.4)
    m = b1 * m + (1 - b1) * g                                     # the pushes, averaged with their signs (T4.3)
    v = b2 * v + (1 - b2) * g * g                                 # their sizes, squared and averaged
    m_hat = m / (1 - b1 ** t)                                     # both start at 0: the corrections (T4.2)
    v_hat = v / (1 - b2 ** t)
    step = m_hat / (v_hat.sqrt() + eps)                           # each push over its own size: about ±1 at step 1 (T4.1)
    return p - lr * step, m, v


def steps_in_rates(pushes, betas=(0.9, 0.999), eps=1e-8, corrected=True):
    """One number's move at each step, in learning rates (weight decay aside), for a list of its pushes."""
    b1, b2 = betas
    m = v = 0.0
    moves = []
    for t, g in enumerate(pushes, 1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        if corrected:
            moves.append(-(m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps))
        else:
            moves.append(-m / (math.sqrt(v) + eps))               # without the corrections
    return moves


def decay_left(rates, weight_decay=0.01):
    """What weight decay alone leaves of a number: each step keeps 1 − weight_decay × that step's learning rate."""
    left = 1.0
    for lr in rates:
        left *= 1 - weight_decay * lr
    return left


def learning_rate(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """train.py's schedule (lr_at): a straight climb to lr over the warm-up, then down a cosine to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup
    done = min(1.0, (step - warmup) / (max_steps - warmup))
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * done))


def first_steps(model, train_ids, steps, rate=learning_rate, B=64, T=32, watch=None):
    """The run's first steps, as train.py takes them (the seed set just before the model was made): each step's learning rate,
    64 random windows, their loss, the way back, then AdamW's step. Calls watch(model) after every step; returns its answers."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    seen = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = rate(step + 1)                          # this step's learning rate
        starts = torch.randint(len(train_ids) - T - 1, (B,))      # this step's 64 places
        x = torch.stack([train_ids[i:i + T] for i in starts])
        y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
        loss = model(x, y)[1]
        opt.zero_grad(set_to_none=True)
        loss.backward()                                           # the way back
        opt.step()                                                # AdamW moves every number
        if watch:
            seen.append(watch(model))
    return seen


@torch.no_grad()
def check_loss(model, ids, starts, T=32):
    """The run's check: the average loss over fixed windows (check_starts), as train.py scores it every 500 steps."""
    model.eval()
    total = sum(model(torch.stack([ids[i:i + T] for i in s]), torch.stack([ids[i + 1:i + T + 1] for i in s]))[1].item() for s in starts)
    model.train()
    return total / len(starts)


# ---------------------------------------------------------------------------------------------------- T5 · the loop

def one_turn(model, opt, train_ids, step, B=64, T=32, clear=True):
    """One turn of train.py's loop, in its order: this step's learning rate, 64 random windows, their loss, the old gradients
    cleared, the way back, then AdamW's step. clear=False leaves the old gradients in (T5.1). Returns the turn's loss."""
    for group in opt.param_groups:
        group["lr"] = learning_rate(step + 1)                     # 1 · this step's learning rate
    starts = torch.randint(len(train_ids) - T - 1, (B,))          # 2 · 64 random places
    x = torch.stack([train_ids[i:i + T] for i in starts])
    y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
    loss = model(x, y)[1]                                         # 3 · their loss
    if clear:
        opt.zero_grad(set_to_none=True)                           # 4 · the last turn's gradients cleared
    loss.backward()                                               # 5 · the way back adds each gradient to what .grad holds
    opt.step()                                                    # 6 · AdamW moves every number
    return loss.item()


def run_turns(model, train_ids, steps, check=None, every=50, clear=True, remake=False):
    """Turns 1 to `steps` from where the model is, scoring check(model) every `every` steps. remake=True makes a new AdamW every
    turn, so its averages and its step count are lost (T5.2). Returns [[step, check], ...]."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    curve = []
    for step in range(steps + 1):
        if check and step % every == 0:
            curve.append([step, check(model)])
        if step == steps:
            return curve
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)  # a new AdamW: its averages start at 0 again
        one_turn(model, opt, train_ids, step, clear=clear)


def what_lasts(model, opt):
    """What carries on from one turn to the next: the learned numbers, AdamW's two averages for each, and its step count."""
    numbers = sum(p.numel() for p in model.parameters())
    averages = sum(s["exp_avg"].numel() + s["exp_avg_sq"].numel() for s in opt.state.values())
    steps = sorted({int(s["step"]) for s in opt.state.values()})
    return numbers, averages, steps


def saved(path):
    """What a saved file holds: each table's name and shape, how many numbers in all, and the file's size in bytes."""
    tables = torch.load(path, weights_only=True)
    shapes = {name: tuple(t.shape) for name, t in tables.items()}
    return shapes, sum(t.numel() for t in tables.values()), os.path.getsize(path)


# ---------------------------------------------------------------------------------------------------- T6 · the check

def other_checks(model, ids, seeds, batches=100, B=64, T=32):
    """The check on other windows: for each seed, its own 100 batches of 64 starts, drawn as train.py draws the check's, scored
    as check_loss scores them. The model's numbers never change: only the windows do."""
    scores = []
    for seed in seeds:
        g = torch.Generator().manual_seed(seed)                   # another seed: other windows
        starts = [torch.randint(len(ids) - T - 1, (B,), generator=g) for _ in range(batches)]
        scores.append(check_loss(model, ids, starts, T))
    return scores                                                 # one score per seed


def how_often_asked(starts, n, T=32):
    """How many characters of a text the check's windows ask about 0 times, once, twice, …: times_asked over those windows,
    counted by times. The first and the last characters are never asked: no window starts before the first or reaches the
    last."""
    asked = times_asked(starts, n, T)                             # [n]   one count per character
    return torch.bincount(asked).tolist()                         # [how many 0 times, once, twice, …]


def loss_by_position(model, ids, T=32):
    """The final score, position by position: answer_costs reads the whole text in windows of 32, side by side; each
    position's costs are averaged down the windows. Position 0 has one character to go on, position 31 all 32."""
    p, _ = answer_costs(model, ids, T)                            # [n·T]  every right answer's probability, window after window
    return (-p.log()).view(-1, T).mean(0)                         # [T]    one average cost per position


def keep_best(checks, margin=1e-4, patience=20):
    """train.py's rule, replayed over a log of checks ([step, held-back score] pairs): a check that beats the best so far by
    more than `margin` keeps a copy, and the wait starts again; any other check adds 1 to the wait; a wait of `patience`
    would stop the run. Returns the kept steps, the step it would stop at (None: never), and the kept copy's score."""
    best, wait, kept = float("inf"), 0, []
    for step, score in checks:
        if score < best - margin:
            best, wait = score, 0                                 # a new best: keep a copy
            kept.append(step)
        elif step:
            wait += 1                                             # no new best: one more check waited
        if patience and wait >= patience:
            return kept, step, best                               # train.py would stop the run here
    return kept, None, best


# ---------------------------------------------------------------------------------------------------- T7 · the curve

def share_of_fall(checks, step):
    """How much of its whole fall a run's held-back check had made at `step`, read from the check at that step: from the first
    check down to the best one, as a share (0: none yet, 1: all of it). `checks` is a log of [step, held-back score] pairs."""
    first, best = checks[0][1], min(score for _, score in checks)
    now = dict(checks)[step]                                      # the check at that step
    return (first - now) / (first - best)


def worse_than_before(checks):
    """The steps whose check scored worse (higher) than the check just before it, in a log of [step, held-back score] pairs."""
    return [step for (_, before), (step, score) in zip(checks, checks[1:]) if score > before]


def gaps(log):
    """The held-back check minus the training check, at every check: [step, gap] pairs, from a log of [step, training score,
    held-back score]. Both are scored at the same moment, on the same numbers: only the text differs."""
    return [[step, val - train] for step, train, val in log]


def write(model, tok, n=1000, seed=42, temperature=1.0):
    """What a model writes from a new line: n characters, picked with its own generator, as the course's writing samples are
    made (training/prompter/unit2.py). The same start, seed and temperature every time, so only the numbers differ."""
    model.eval()
    with torch.no_grad():
        out = model.generate(torch.tensor([tok.encode("\n")]), n, temperature, torch.Generator().manual_seed(seed))
    return tok.decode(out[0, 1:].tolist())                        # the new line it started from left out


def word_share(text, training_text):
    """The share of a text's words found in the training text, and the words found nowhere in it. A word is a run of letters
    and apostrophes, read in lower case. Found means spelt somewhere in the training text: it says nothing about sense."""
    known = set(re.findall(r"[a-z']+", training_text.lower()))   # every word the training text spells
    words = re.findall(r"[a-z']+", text.lower())
    found = sum(w in known for w in words)
    return found / len(words), sorted({w for w in words if w not in known})

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # the whole text as ids
cut = int(0.9 * len(ids))
train_ids, val_ids = ids[:cut], ids[cut:]                   # the first 90% trains; the rest is held back
_, val_starts = check_starts(len(train_ids), len(val_ids))  # the run's held-back windows, the same every check
check = lambda m: check_loss(m, val_ids, val_starts)        # the held-back check, as the run's
space = tok.chars.index(' ')

# train.py's order: the seed, then the model (its starting numbers), then the turns
torch.manual_seed(1337)
model0 = Prompter(cfg)
starts = torch.randint(len(train_ids) - 32 - 1, (64,))
assert starts.tolist() == [160204, 793724, 810574, 964722, 40734, 423895, 818722, 483307, 510490, 928356, 614059, 30062, 737496, 407796, 59365, 998494, 864171, 929951, 942896, 637372, 408276, 289366, 811280, 165163, 748492, 428999, 260643, 560439, 371836, 390719, 776246, 678592, 338694, 316420, 711307, 461584, 554475, 251253, 546994, 358001, 64331, 976218, 589122, 206049, 243936, 44076, 936280, 819204, 162295, 566156, 935951, 895342, 457178, 537830, 274346, 823229, 14080, 899619, 528867, 495983, 90293, 96640, 21128, 975278]   # the recorded run's first batch
print(f"{count_params(model0):,} learned numbers · the held-back check at step 0: {check(model0):.4f} (the run's log: 4.1780)")
assert count_params(model0) == 9417

9,417 learned numbers · the held-back check at step 0: 4.1780 (the run's log: 4.1780)


# Section 1 · One turn, in order

One turn of `train.py`'s loop, line by line, and what carries on from one turn to the next. Each chapter below follows its page:
Picture it, Watch it in our run, then Read and run the code.

## T5.1 · Six lines, one turn

*What does one turn of the loop do, and in what order?*

**Predict first** (the page asks it too, before its clip): each turn, `train.py` clears every number's gradient just before the way back: `opt.zero_grad`. Without that line, what would the
space's gradient read at step 2: step 1's gradient and step 2's, added together; only step 2's, as the way back writes over the old
one; or 0, as AdamW's step uses the gradient up?

### T5.1.1 · Picture it

One turn uses every station opened so far. In what order, and where does each gradient wait? One turn is six lines: the learning rate, the windows, their loss,
clearing, the way back, then the step. Picture one made-up number. The way back leaves its gradient, 2, in a slot, and the step
reads it from there. Clearing empties every slot first. Only then does the way back fill them. (The page's number is made up to
show the idea.)

### T5.1.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's first turns, replayed from its seed: as `train.py` ran them, and never cleared. Cleared, the space's gradient was
about −0.14 each turn. Never cleared, it read −0.284 at step 2, and −0.425 at step 3. The held-back loss
rose to about 5 by step 500, and about 10 by step 1,000. The run's was
2.12. The way back adds each gradient to what the slot holds. So the old ones must go first.

### T5.1.3 · Read and run the code

`one_turn` in `code/training_steps.py` (copied in above) is one turn of `train.py`'s loop. Its six lines run in the run's order: the
rate, the windows, the loss, clearing, the way back, the step. The cell runs it three times with `clear=False`: the space's gradient
reads −0.284 after two turns, piled up. That is because `loss.backward` adds each gradient to `.grad`; `opt.zero_grad`
empties it, just before. Then `run_turns` replays 1,000 turns never cleared, and the run's own loop beside them.

A loop that piles up its gradients magnifies the last digits of its sums, and those change with how many threads add them up. This
notebook uses 4, as the run did. On 1, 4 and 8 threads, never cleared reads 10.60, 10.83, 9.89 at step 1,000, so the page
says about 10.

In [7]:
torch.manual_seed(1337)
model = Prompter(cfg)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
piled = []
for step in range(3):                                       # three turns, the old gradients never cleared
    one_turn(model, opt, train_ids, step, clear=False)
    piled.append(model.head_bias.grad[space].item())
print("the space's gradient, never cleared, after turns 1, 2, 3:", [round(g, 3) for g in piled])
assert all(abs(a - b) < 1e-4 for a, b in zip(piled, [-0.14435, -0.28384, -0.42526]))

runs = {}
for name, clear in (('as the run', True), ('never cleared', False)):
    torch.manual_seed(1337)
    runs[name] = dict(run_turns(Prompter(cfg), train_ids, 1000, check=check, every=500, clear=clear))
    print(f"{name:>13}: the held-back check at step 500 {runs[name][500]:.2f} · at step 1,000 {runs[name][1000]:.2f}")
assert abs(runs['as the run'][500] - 2.30359) < 1e-3 and abs(runs['as the run'][1000] - 2.12428) < 1e-3   # the run's log
print("never cleared is magnified from the sums' last digits, which change with the thread count: the page's “about” is the average of 1, 4 and 8 threads")
assert 4.644 < runs['never cleared'][500] < 5.320 and 8.391 < runs['never cleared'][1000] < 12.330

the space's gradient, never cleared, after turns 1, 2, 3: [-0.144, -0.284, -0.425]


   as the run: the held-back check at step 500 2.30 · at step 1,000 2.12


never cleared: the held-back check at step 500 4.96 · at step 1,000 10.83
never cleared is magnified from the sums' last digits, which change with the thread count: the page's “about” is the average of 1, 4 and 8 threads


### ✋ Your turn

Never cleared, the slot after turn 3 holds three turns' gradients. How big was turn 3's own push? Take turn 2's total from turn
3's, and put it in `answer`.

In [8]:
answer = None   # TODO: piled[2] - piled[1]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (piled[2] - piled[1])) < 1e-12, "turn 3's total minus turn 2's: piled[2] - piled[1]"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
answer = piled[2] - piled[1]
print(f"turn 3's own push: {answer:.3f}, about the same as turn 1's, {piled[0]:.3f}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (piled[2] - piled[1])) < 1e-12, "turn 3's total minus turn 2's: piled[2] - piled[1]"
    print("✓ right:", answer)

turn 3's own push: -0.141, about the same as turn 1's, -0.144
✓ right: -0.14141395688056946


## T5.2 · What lasts from turn to turn

*What carries on from one turn to the next?*

**Predict first** (the page asks it too, before its clip): suppose `train.py` made a new AdamW at every turn, keeping the model's numbers. A new AdamW starts its averages at 0 and counts its
steps from 1. How would the run's steps change: every step would be a first step, nearly every number moving the full learning
rate; not at all, as AdamW only turns this turn's gradient into a step; or no number would move, as averages at 0 give steps of 0?

### T5.2.1 · Picture it

Every turn draws new windows. What does the next turn get from this one? Picture three made-up turns. Each makes its own windows,
its own loss and its own gradients, then drops them. Kept: the number, and AdamW's averages for it. With pushes of 1, the push
average reads 0.1, then 0.19, then 0.271. So each turn starts where the last one stopped. (The page's turns are made up to show
the idea.)

### T5.2.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's first 1,000 turns again: as the run, and with a new AdamW made every turn. At step 500, AdamW moved
2 of the 9,417 numbers the full learning rate. Remade every turn, it moved about 9,400. Every step
was a first step. It still learned, more slowly: about 2.23 at step 1,000, against the run's
2.12. AdamW's averages are its memory: they carry on from turn to turn, with the numbers.

### T5.2.3 · Read and run the code

`what_lasts` counts what carries on: after one turn, 9,417 numbers, 18,834 averages, two for each, and the step
count, 1. `run_turns`, given `remake=True`, makes a new AdamW before every turn; everything else is the same turn, `one_turn`. The cell
writes that loop out, so it can count the numbers that moved the full rate at step 500, as the run and remade. Remade, that count
changes with the thread count: 9,366, 9,376, 9,384 on 1, 4 and 8 threads, so the page says about 9,400.

In [10]:
torch.manual_seed(1337)
model = Prompter(cfg)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
one_turn(model, opt, train_ids, 0)
print('after one turn, (numbers, averages, step counts):', what_lasts(model, opt))
assert what_lasts(model, opt) == (9417, 18834, [1])

flat = lambda m: torch.cat([p.detach().ravel() for p in m.parameters()])
def replay(remake, steps):
    """The first `steps` turns from the seed, a new AdamW every turn or not; and how many numbers moved the full rate at step 500."""
    torch.manual_seed(1337)
    model = Prompter(cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for step in range(steps):
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)   # a new AdamW: its averages at 0, its step count at 1
        if step == 499:
            before = flat(model).clone()
        one_turn(model, opt, train_ids, step)
        if step == 499:
            lr = learning_rate(500)
            own = (before * (1 - lr * 0.01) - flat(model)).abs()   # each step, the pull toward 0 aside
            full = int(((own - lr).abs() <= 0.01 * lr).sum())
    return model, full

_, full_run = replay(False, 500)
remade, full_remade = replay(True, 1000)
print(f"step 500, moved the full rate: as the run {full_run:,} · remade every turn {full_remade:,} (of 9,417; 4 threads, as the run)")
print(f"remade every turn, the held-back check at step 1,000: {check(remade):.2f} (the run's log: 2.12)")
assert full_run <= 10 and abs(full_remade - 9376) <= 50
assert 2.187 < check(remade) < 2.266

after one turn, (numbers, averages, step counts): (9417, 18834, [1])


step 500, moved the full rate: as the run 2 · remade every turn 9,376 (of 9,417; 4 threads, as the run)


remade every turn, the held-back check at step 1,000: 2.23 (the run's log: 2.12)


### ✋ Your turn

Does AdamW's step count carry on too? The lines make a fresh model and AdamW and run three turns. Put `what_lasts(m, o)[2]`, the
step count, in `answer`.

In [11]:
torch.manual_seed(1337)
m = Prompter(cfg)
o = torch.optim.AdamW(m.parameters(), lr=3e-3)
for step in range(3):
    one_turn(m, o, train_ids, step)
answer = None   # TODO: what_lasts(m, o)[2]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [3], 'the step count after three turns: what_lasts(m, o)[2]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
torch.manual_seed(1337)
m = Prompter(cfg)
o = torch.optim.AdamW(m.parameters(), lr=3e-3)
for step in range(3):
    one_turn(m, o, train_ids, step)
answer = what_lasts(m, o)[2]
print(f"AdamW's step count after 3 turns: {answer[0]}: it carried on from turn to turn")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [3], 'the step count after three turns: what_lasts(m, o)[2]'
    print("✓ right:", answer)

AdamW's step count after 3 turns: 3: it carried on from turn to turn
✓ right: [3]


# Section 2 · The whole run

The run's end, and the run again: what `train.py` saved, and how close a second seed lands.

## T5.3 · What the run saved

*What did the run leave behind?*

**Predict first** (the page asks it too, before its clip): at the end, `train.py` saves the model in one file, best.pt: 42,603 bytes. One number takes 4 bytes. What does it hold: the
9,417 learned numbers, and no others; the learned numbers and AdamW's 18,834 averages, ready to carry on; or the
numbers at every check, the whole run step by step?

### T5.3.1 · Picture it

When a run stops, what should it write down? Picture a made-up model of 6 numbers, saved to a file. Each number takes 4 bytes, so
the numbers fill 24 bytes. Their names, and the file's own layout, add some more. So a file's size says roughly what is in it.
(The page's model is made up to show the idea.)

### T5.3.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is best.pt, the file `train.py` saved at the end, opened. It holds 17 tables, 9,417 numbers
in all: 37,668 bytes of numbers; their names and the file's own layout fill the rest. Not in it: AdamW's
18,834 averages, its step count, or the random stream's place. They are the numbers at the run's best check, step 98,000, not its last step.

### T5.3.3 · Read and run the code

The function `saved` opens a saved file. It reads each table's name and shape, and counts its numbers. On best.pt:
17 tables, 9,417 numbers, 42,603 bytes. `torch.load` gives back one table for each
name, just as `train.py` saved them. The cell lists them, and loads them into a model.

In [13]:
paths = [os.path.join(here, '..', 'model', 'prompter-r1-best.pt'),                                              # in the labs
         os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'results', 'prompter-r1', 'best.pt')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the file the run saved is not here: run the Setup cell at the top"
shapes, numbers, size = saved(found[0])
for name, shape in shapes.items():
    print(f"{name:>26}  {' × '.join(map(str, shape)):>8}")
print(f"{len(shapes)} tables · {numbers:,} numbers · {4 * numbers:,} bytes of numbers, of {size:,}")
best = Prompter(cfg)
best.load_state_dict(torch.load(found[0], weights_only=True))   # every table has its place in the model: nothing more, nothing less
print(f"the held-back check on best.pt: {check(best):.4f} (the run's best check, step 98,000: 1.9351)")
assert abs(check(best) - 1.9350941205024719) < 1e-4
assert (len(shapes), numbers, size) == (17, 9417, 42603)
assert not any('exp_avg' in k or 'step' in k for k in shapes)   # no averages, no step count

                 head_bias        65
            tok_emb.weight   65 × 28
            pos_emb.weight   32 × 28
       blocks.0.ln1.weight        28
         blocks.0.ln1.bias        28
  blocks.0.attn.qkv.weight   84 × 28
    blocks.0.attn.qkv.bias        84
 blocks.0.attn.proj.weight   28 × 28
   blocks.0.attn.proj.bias        28
       blocks.0.ln2.weight        28
         blocks.0.ln2.bias        28
      blocks.0.ff.0.weight   56 × 28
        blocks.0.ff.0.bias        56
      blocks.0.ff.2.weight   28 × 56
        blocks.0.ff.2.bias        28
               ln_f.weight        28
                 ln_f.bias        28
17 tables · 9,417 numbers · 37,668 bytes of numbers, of 42,603


the held-back check on best.pt: 1.9351 (the run's best check, step 98,000: 1.9351)


### ✋ Your turn

Which of best.pt's tables holds the most numbers? Find it in `shapes`, and put its name in `answer`.

In [14]:
answer = None   # TODO: max(shapes, key=lambda k: math.prod(shapes[k]))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == max(shapes, key=lambda k: math.prod(shapes[k])), "each table's numbers: math.prod(shape); the biggest: max(shapes, key=...)"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
answer = max(shapes, key=lambda k: math.prod(shapes[k]))
print(answer, shapes[answer], f'{math.prod(shapes[answer]):,} numbers')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == max(shapes, key=lambda k: math.prod(shapes[k])), "each table's numbers: math.prod(shape); the biggest: max(shapes, key=...)"
    print("✓ right:", answer)

blocks.0.attn.qkv.weight (84, 28) 2,352 numbers
✓ right: blocks.0.attn.qkv.weight


## T5.4 · Running it twice

*What would running it again give?*

**Predict first** (the page asks it too, before its clip): replayed from seed 1337, the first 5,000 steps met the run's log at all 11 checks. A second run, seed
1338, starts from other numbers and draws other windows. At step 5,000, how far apart are their held-back checks: about a
hundredth or two; not at all, as the recipe decides everything; or about 0.5, as other windows make another model?

### T5.4.1 · Picture it

One seed made our run. What would another seed make? Picture a made-up seed, 7. It draws three numbers: 4, 1, 9. Seed 7 again draws
4, 1, 9: the same draws, every time. Seed 8 draws others, 2, 6, 3, and every draw after them changes too. (The page's seeds and
draws are made up to show the idea.)

### T5.4.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is the run's held-back check, steps 500 to 5,000, replayed from its seed. The replay met the run's log at all
11 checks, step 0 included. A second run, its seed 1 more, starts from other numbers and draws other windows. At step
5,000: 2.032 against our 2.017. Just 0.015 apart.

### T5.4.3 · Read and run the code

`run_turns` runs turn after turn. Every few steps, it scores the model, and keeps the scores as a curve. The cell first runs the
run's first 100 turns twice from its seed: the same losses, turn for turn. Then it seeds a second run, 1338, and checks it
every 500 steps, beside the run's own log. Each turn is still `one_turn`: only the seed changed. (5,000 turns take a minute or
two.) The same seed repeats a run only with the same number of threads: the run used 4, and so does this notebook.

In [16]:
def losses(seed, turns):
    """The loss of each of the first `turns` turns from `seed`."""
    torch.manual_seed(seed)
    model = Prompter(cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    return [one_turn(model, opt, train_ids, step) for step in range(turns)]
print('the same seed twice, 100 turns: the same losses?', losses(1337, 100) == losses(1337, 100))

LOG = {0: 4.17804, 500: 2.30359, 1000: 2.12428, 1500: 2.08149, 2000: 2.05315, 2500: 2.04148, 3000: 2.0435, 3500: 2.03529, 4000: 2.03394, 4500: 2.01746, 5000: 2.01698}   # the run's log: the held-back check every 500 steps
torch.manual_seed(1338)                                 # a second run: its seed 1 more
curve = run_turns(Prompter(cfg), train_ids, 5000, check=check, every=500)
for step, v in curve:
    print(f"step {step:>5,}: seed 1338 {v:.3f} · the run's log {LOG[step]:.3f}")
gap = curve[-1][1] - LOG[5000]
print(f"at step 5,000: {gap:.3f} apart")
assert abs(curve[-1][1] - 2.03211) < 0.005 and abs(gap - 0.01513) < 0.005

the same seed twice, 100 turns: the same losses? True


step     0: seed 1338 4.178 · the run's log 4.178
step   500: seed 1338 2.291 · the run's log 2.304
step 1,000: seed 1338 2.136 · the run's log 2.124
step 1,500: seed 1338 2.087 · the run's log 2.081
step 2,000: seed 1338 2.073 · the run's log 2.053
step 2,500: seed 1338 2.060 · the run's log 2.041
step 3,000: seed 1338 2.057 · the run's log 2.043
step 3,500: seed 1338 2.057 · the run's log 2.035
step 4,000: seed 1338 2.043 · the run's log 2.034
step 4,500: seed 1338 2.035 · the run's log 2.017
step 5,000: seed 1338 2.032 · the run's log 2.017
at step 5,000: 0.015 apart


### ✋ Your turn

At step 0 both runs score about the even guess: their starting numbers are small, whatever the seed. After that, at which check
were the two runs closest? Find the step in `curve` (after step 0) whose score is nearest the run's log, and put it in `answer`.

In [17]:
answer = None   # TODO: min(curve[1:], key=lambda r: abs(r[1] - LOG[r[0]]))[0]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == min(curve[1:], key=lambda r: abs(r[1] - LOG[r[0]]))[0], 'skip step 0: curve[1:]; nearest: min(..., key=lambda r: abs(r[1] - LOG[r[0]])); its step: [0]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [18]:
#@title Show a solution { display-mode: "form" }
answer = min(curve[1:], key=lambda r: abs(r[1] - LOG[r[0]]))[0]
print(f'closest at step {answer:,}')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == min(curve[1:], key=lambda r: abs(r[1] - LOG[r[0]]))[0], 'skip step 0: curve[1:]; nearest: min(..., key=lambda r: abs(r[1] - LOG[r[0]])); its step: [0]'
    print("✓ right:", answer)

closest at step 1,500
✓ right: 1500


One turn of `train.py`'s loop is six lines: the learning rate, 64 windows, their loss, the old gradients cleared, the way back,
then AdamW's step. The way back adds to what is there: never cleared, the held-back loss rose to about 5 by
step 500. The learned numbers and AdamW's 18,834 averages carry on; remade every turn, AdamW moved about 9,400
numbers the full rate at step 500, against the run's 2. At the end, `train.py` saved best.pt: the 9,417 learned
numbers of its best check, step 98,000, and no other numbers. On the same computer, the same seed repeats the run; a
second seed landed 0.015 away at step 5,000.

Next: **T6 · the check**, on the siding every 500 steps.